In [1]:
from pathlib import Path
import time
 
import numpy as np
from PIL import Image
from scipy.signal import convolve2d

from conv2d_driver import Conv2DDriver



In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────────────────────────────────────
BITSTREAM        = "conv100m.bit"
IMAGE_FOLDER     = "./testSet"
LOG_FILE         = "testbench_log.txt"
 
KERNEL = np.array([
    [1,  0, -1],
    [2,  0, -2],
    [1,  0, -1],
], dtype=np.int32)
 
# Width must be a multiple of 4 (pixel packing), min 16; height min 16.
# Start small for bring-up; widen once everything passes.
# TEST_WIDTHS  = [16, 32, 60]
# TEST_HEIGHTS = [16, 32, 60]
# Full sweep — uncomment once bring-up is stable:
TEST_WIDTHS  = list(range(16, 61, 4))
TEST_HEIGHTS = list(range(16, 61)) + [128, 512, 1020]
 
MAX_W            = 60
MAX_H            = 1020
 
VALID_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
 
RUN_SYNTHETIC   = True
RUN_IMAGE_TESTS = True
 
 
# ─────────────────────────────────────────────────────────────────────────────
# Reference model
# ─────────────────────────────────────────────────────────────────────────────
def reference_model(image_2d: np.ndarray, kernel_2d: np.ndarray) -> np.ndarray:
    ref = convolve2d(image_2d.astype(np.int32), kernel_2d.astype(np.int32), mode="valid")
    return np.clip(ref, 0, 255).astype(np.uint8)
 
 
# ─────────────────────────────────────────────────────────────────────────────
# Output comparison  (writes to log only)
# ─────────────────────────────────────────────────────────────────────────────
def compare_outputs(
    expected:  np.ndarray,
    actual:    np.ndarray,
    test_name: str,
    log,
) -> bool:
    if expected.shape != actual.shape:
        log(f"  FAIL [{test_name}] shape mismatch "
            f"expected={expected.shape} actual={actual.shape}")
        return False
 
    if np.array_equal(expected, actual):
        log(f"  PASS [{test_name}]")
        return True
 
    mismatch = np.argwhere(expected != actual)
    r, c     = mismatch[0]
    n_diff   = len(mismatch)
    max_err  = int(np.max(np.abs(
        expected.astype(np.int32) - actual.astype(np.int32)
    )))
    log(f"  FAIL [{test_name}] {n_diff} pixels differ, "
        f"first=({r},{c}) exp={expected[r,c]} got={actual[r,c]}, "
        f"max_err={max_err}")
    return False
 
 
# ─────────────────────────────────────────────────────────────────────────────
# Synthetic test cases  (lazy generator — no upfront allocation)
# ─────────────────────────────────────────────────────────────────────────────
def iter_synthetic_cases(heights, widths, kernel_shape):
    kh, kw = kernel_shape
    rng    = np.random.default_rng(seed=42)
 
    for h in heights:
        for w in widths:
            if h < kh or w < kw:
                continue
            yield f"zeros_{h}x{w}",   np.zeros((h, w), dtype=np.uint8)
            yield f"ones_{h}x{w}",    np.full((h, w), 128, dtype=np.uint8)
            yield f"ramp_{h}x{w}",    (np.arange(h * w, dtype=np.int32) % 256).reshape(h, w).astype(np.uint8)
            yield f"random_{h}x{w}",  rng.integers(0, 256, size=(h, w), dtype=np.uint8)
            imp = np.zeros((h, w), dtype=np.uint8)
            imp[h // 2, w // 2] = 255
            yield f"impulse_{h}x{w}", imp
 
 
# ─────────────────────────────────────────────────────────────────────────────
# Image loading
# ─────────────────────────────────────────────────────────────────────────────
def load_grayscale_image(path: Path) -> np.ndarray:
    return np.array(Image.open(path).convert("L"), dtype=np.uint8)
 
 
# ─────────────────────────────────────────────────────────────────────────────
# Test runner
# ─────────────────────────────────────────────────────────────────────────────
def run_tests():
    total   = 0
    passed  = 0
    t_start = time.time()
 
    with open(LOG_FILE, "w") as f:
 
        def log(msg: str = ""):
            f.write(msg + "\n")
            f.flush()
 
        log("=" * 70)
        log(" Conv2D Accelerator Test Bench")
        log("=" * 70)
 
        with Conv2DDriver(BITSTREAM, KERNEL, max_w=MAX_W, max_h=MAX_H) as drv:
            kh, kw = KERNEL.shape
 
            # ── Synthetic tests ────────────────────────────────────────────
            if RUN_SYNTHETIC:
                log("\n--- Synthetic tests ---")
                t_sect  = time.time()
                n_cases = 0
 
                for name, img in iter_synthetic_cases(TEST_HEIGHTS, TEST_WIDTHS, KERNEL.shape):
                    try:
                        actual = drv.convolve(img)
                    except Exception as e:
                        log(f"  ERROR [{name}]: {type(e).__name__}: {e}")
                        total   += 1
                        n_cases += 1
                        continue
 
                    expected = reference_model(img, KERNEL)
                    ok       = compare_outputs(expected, actual, name, log)
                    total   += 1
                    passed  += ok
                    n_cases += 1
 
                log(f"\n  {n_cases} cases in {time.time() - t_sect:.2f}s")
 
            # ── Image crop tests ───────────────────────────────────────────
            if RUN_IMAGE_TESTS:
                log("\n--- Image crop tests ---")
                folder = Path(IMAGE_FOLDER)
                image_files = sorted(
                    p for p in folder.iterdir()
                    if p.suffix.lower() in VALID_EXTENSIONS
                ) if folder.is_dir() else []
 
                if not image_files:
                    log(f"  No images found in '{IMAGE_FOLDER}', skipping.")
                else:
                    for image_path in image_files:
                        full_image   = load_grayscale_image(image_path)
                        img_h, img_w = full_image.shape
                        log(f"\n  {image_path.name}  ({img_h}x{img_w})")
 
                        t_img = time.time()
                        n_img = 0
                        for h in TEST_HEIGHTS:
                            if h < kh or h > img_h or h > MAX_H:
                                continue
                            for w in TEST_WIDTHS:
                                if w < kw or w > img_w or w > MAX_W:
                                    continue
 
                                crop     = full_image[:h, :w]
                                expected = reference_model(crop, KERNEL)
                                try:
                                    actual = drv.convolve(crop)
                                except Exception as e:
                                    log(f"    ERROR [{image_path.name} {h}x{w}]: "
                                        f"{type(e).__name__}: {e}")
                                    total += 1
                                    continue
 
                                ok     = compare_outputs(
                                    expected, actual,
                                    f"{image_path.name} {h}x{w}",
                                    log,
                                )
                                total  += 1
                                passed += ok
                                n_img  += 1
 
                        log(f"    {n_img} crops in {time.time() - t_img:.2f}s")
 
        # ── Summary ────────────────────────────────────────────────────────
        elapsed = time.time() - t_start
        status  = "PASS" if passed == total else "FAIL"
        ms_each = 1000 * elapsed / total if total else 0
 
        summary = (
            "\n" + "=" * 70 + "\n"
            f" Result  : {status}\n"
            f" Passed  : {passed}/{total}\n"
            f" Time    : {elapsed:.2f}s  ({ms_each:.1f} ms/test)\n"
            f" Details : {LOG_FILE}\n"
            + "=" * 70
        )
        log(summary)
 
    # Console only gets the final summary
    print(summary)
 


In [3]:
if __name__ == "__main__":
    run_tests()



 Result  : PASS
 Passed  : 1458880/1458880
 Time    : 4206.86s  (2.9 ms/test)
 Details : testbench_log.txt
